# Preprocessing (ML Dataset)

This notebook uses `youtube_shorts_tiktok_trends_2025.csv_ML.csv` directly. It skips the original-vs-ML CSV comparison and saves a separate processed dataset for the ML notebook copies.


## Load Data


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../data/kaggleData/youtube_shorts_tiktok_trends_2025.csv_ML.csv")

print("ML dataset shape:", df.shape)
df.head()


## Target Distribution

Check the class balance in the ML-ready dataset before splitting.


In [ ]:
display(df["trend_label"].value_counts(normalize=True).sort_index())


## Basic Data Checks


In [ ]:
summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "n_unique": df.nunique(),
    }
)

print("Duplicate rows:", df.duplicated().sum())
display(summary.sort_values("missing", ascending=False))


## Clean Values

The ML CSV is already feature engineered, so preprocessing here is limited to consistent string cleanup before encoding.


In [ ]:
clean = df.copy()

text_columns = clean.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    clean[col] = clean[col].astype("string").str.strip().str.lower()

clean.head()


## Choose Columns

Use the ML-ready columns as features and remove only the target label.


In [ ]:
TARGET = "trend_label"

X = clean.drop(columns=[TARGET])
y = clean[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())


## Feature Groups

Use these groups later for RQ experiments. The group definitions are written once here and saved with the processed files.


In [ ]:
feature_groups = {
    "metadata": [
        "platform",
        "region",
        "language",
        "category",
        "traffic_source",
        "device_brand",
        "platform_cat",
        "region_cat",
        "language_cat",
        "category_cat",
        "traffic_source_cat",
        "device_brand_cat",
    ],
    "content_basic": [
        "title_len",
        "text_richness",
    ],
    "creator": [
        "creator_tier",
        "creator_tier_cat",
    ],
    "engagement_observed": [
        "like_rate",
        "comment_rate",
        "share_rate",
        "like_rate_log",
        "comment_rate_log",
        "share_rate_log",
        "views_per_day",
        "likes_per_day",
        "rel_like",
        "rel_share",
        "rel_combo",
    ],
    "interactions": [
        "like_hashtag_interaction",
        "share_hashtag_interaction",
        "richness_traffic_interaction",
        "weekend_hashtag_boost",
    ],
}

feature_groups = {
    group: [col for col in cols if col in X.columns]
    for group, cols in feature_groups.items()
}

display(pd.DataFrame([
    {"feature_group": group, "n_raw_columns": len(cols), "raw_columns": ", ".join(cols)}
    for group, cols in feature_groups.items()
]))


## Train / Validation / Test Split

The split is stratified so each set keeps roughly the same class balance.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

display(y_train.value_counts(normalize=True).sort_index())
display(y_val.value_counts(normalize=True).sort_index())
display(y_test.value_counts(normalize=True).sort_index())


## Encode Features

Categorical columns are converted to dummy variables here, so RQ1, RQ2, and RQ3 can read numeric CSVs directly without building preprocessing pipelines again.


In [ ]:
categorical_columns = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_columns = [col for col in X_train.columns if col not in categorical_columns]

# Boolean columns are model-ready once converted to 0/1 values.
bool_columns = X_train[numeric_columns].select_dtypes(include=["bool"]).columns.tolist()
for data_part in [X_train, X_val, X_test]:
    for col in bool_columns:
        data_part[col] = data_part[col].astype("int8")

numeric_medians = X_train[numeric_columns].median(numeric_only=True)

def encode_split(X_part):
    numeric_part = X_part[numeric_columns].fillna(numeric_medians)
    categorical_part = pd.get_dummies(
        X_part[categorical_columns],
        dtype="int8",
    )
    return pd.concat([numeric_part, categorical_part], axis=1)

X_train_encoded = encode_split(X_train)
X_val_encoded = encode_split(X_val).reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = encode_split(X_test).reindex(columns=X_train_encoded.columns, fill_value=0)

X_train = X_train_encoded
X_val = X_val_encoded
X_test = X_test_encoded

print("Categorical columns encoded:", len(categorical_columns))
print("Numeric columns kept:", len(numeric_columns))
print("Encoded train shape:", X_train.shape)
print("Encoded validation shape:", X_val.shape)
print("Encoded test shape:", X_test.shape)

display(X_train.head())


## Save


In [ ]:
processed_dir = Path("../data/processed_ml")
processed_dir.mkdir(parents=True, exist_ok=True)

raw_to_group = {
    column: group
    for group, columns in feature_groups.items()
    for column in columns
}

feature_metadata_rows = []
for encoded_feature in X_train.columns:
    original_feature = encoded_feature
    for categorical_column in categorical_columns:
        if encoded_feature.startswith(f"{categorical_column}_"):
            original_feature = categorical_column
            break

    feature_metadata_rows.append({
        "feature": encoded_feature,
        "original_feature": original_feature,
        "feature_group": raw_to_group.get(original_feature, "other"),
    })

feature_metadata = pd.DataFrame(feature_metadata_rows)
feature_group_table = feature_metadata[["feature_group", "feature"]].copy()

train = X_train.copy()
train[TARGET] = y_train

validation = X_val.copy()
validation[TARGET] = y_val

test = X_test.copy()
test[TARGET] = y_test

train.to_csv(processed_dir / "train.csv", index=False)
validation.to_csv(processed_dir / "validation.csv", index=False)
test.to_csv(processed_dir / "test.csv", index=False)
feature_metadata.to_csv(processed_dir / "feature_metadata.csv", index=False)
feature_group_table.to_csv(processed_dir / "feature_groups.csv", index=False)

print("Saved files to", processed_dir)
print("Saved feature metadata:", feature_metadata.shape)
display(feature_metadata.head())


## Summary

This ML preprocessing notebook uses `youtube_shorts_tiktok_trends_2025.csv_ML.csv` directly and skips the original-vs-ML CSV comparison from the original preprocessing workflow. The ML dataset contains 50,000 rows and 32 columns total, including the target label, with 31 input columns before encoding. Its target distribution is more imbalanced than the original CSV: `stable` is the majority class at about 55%, `rising` is about 25%, and `declining` and `seasonal` are each about 9-10%.

The notebook creates a stratified train/validation/test split with 35,000 training rows, 7,500 validation rows, and 7,500 test rows. After encoding categorical features, the processed ML feature matrix has 72 model-ready columns. These outputs are saved separately under `../data/processed_ml`, so the ML experiments do not overwrite the original processed files in `../data/processed`.

The feature groups saved for the ML notebooks are `metadata`, `content_basic`, `creator`, `engagement_observed`, and `interactions`. This differs from the original preprocessing, where temporal and raw engagement fields dominated the feature layout. The new `interactions` group is especially important in later notebooks because it captures engineered signals such as hashtag and weekend interaction effects.